<a href="https://colab.research.google.com/github/Harshith493/fkyrank_ml/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

Lane 1 -- Ranking Signal Analysis.

I'm starting here because before I commit to scoring individual pages (Lane 2) or building archetypes (Lane 3), I want evidence for which signals in this data actually associate with visibility, clicks, or engagement -- rather than guessing. The starter dataset has 44 columns spanning content properties (word count, freshness, content type), search performance (position, CTR, impressions), and engagement (sessions, scroll rate) across 32 clients and 30,000 pages. That breadth is exactly what a signal-analysis pass needs: enough variety to compare levers against each other before picking one to build a ranking or scoring system around in later weeks. Lane 1's output (a signal report with evidence-backed recommendations) also gives every later lane I might touch a grounded starting point instead of an assumption.

In [6]:
import pandas as pd
from pathlib import Path
import sys, subprocess

LANE = "Lane 1 -- Ranking Signal Analysis"
print(f"Chosen lane: {LANE}")

REPO_URL = "https://github.com/Harshith493/fkyrank_ml.git"
REPO_NAME = "fkyrank_ml"

if "google.colab" in sys.modules:
    # Running in Colab: the repo isn't on disk yet, so clone it once.
    if not Path(REPO_NAME).exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    DATA_PATH = Path(REPO_NAME) / "data" / "raw" / "content_refresh_anonymized.csv"
else:
    # Running locally: find the repo root by walking up from the current
    # working directory until the data file turns up, so this cell works
    # no matter where the kernel's working directory happens to be.
    here = Path.cwd()
    DATA_PATH = None
    for candidate in [here, *here.parents]:
        p = candidate / "data" / "raw" / "content_refresh_anonymized.csv"
        if p.exists():
            DATA_PATH = p
            break
    if DATA_PATH is None:
        raise FileNotFoundError("Could not locate data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns from {DATA_PATH.name}")


Chosen lane: Lane 1 -- Ranking Signal Analysis
Loaded 30,000 rows x 44 columns from content_refresh_anonymized.csv


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

Search question: Which content and search signals -- freshness, word count, position, content type -- are actually associated with visibility, clicks, or engagement in this data, and which ones turn out to be noise?

Unit of analysis: a single page (content_id), described by its content properties and its 90-day search/engagement performance. Not a client, and not a day -- the starter data is one row per page, already aggregated over a 90-day window.

Output: a short signal report -- a ranked list of candidate signals with the direction and rough size of their association with performance, plus which candidates turned out weaker than expected -- written for someone who won't read the code.

Decision this improves: which content lever -- freshness, depth/word count, position/CTR fixes, or content type -- an SEO/content strategy lead should prioritize as a broad focus for their team's limited editorial time, before any team spends effort building or acting on a specific per-page system.

Who acts, and how: a content/SEO strategy lead deciding where to point a small editorial team across many clients' inventories -- not a reviewer deciding on one page (that's Lane 2's job). They act by choosing which lever to invest in broadly: e.g. "run a freshness sprint" vs "run a meta-description/CTR sprint" vs "expand thin content."

Cost of a wrong call: if the evidence says position/CTR is the strongest lever but the team is told to prioritize word-count expansion instead, editorial hours get spent rewriting pages that were never going to move the needle, while pages with real, cheap fixes (title/meta, position-adjacent CTR issues) go untouched. The reverse mistake -- dismissing a lever that actually matters -- means missing real wins across many clients at once, not just one page.

Why data/ML helps, and why this isn't "just train a model": the candidate signals are confounded with each other (content type, freshness, and position all interact, as the group-count check below shows), and at 30,000 rows across 32 clients, eyeballing which one is doing the real work isn't tractable by hand. This week's job is disciplined EDA -- grouped comparisons and correlations to separate real association from noise -- not fitting a predictive model. A model would be premature before I know which signals are even worth feeding it; that decision belongs to a later week (ML-05/ML-06 baseline and modeling), once this signal audit has narrowed the field.

In [7]:
# Ground the framing in scale: how many clients, content types, and tier
# combinations are we really asking someone to reason about by eye?
n_clients = df["client_id"].nunique()
n_content_types = df["content_type"].nunique()
n_tier_combos = df.groupby(["content_type", "position_tier", "freshness_tier"]).ngroups

print(f"Clients in this slice: {n_clients}")
print(f"Content types: {n_content_types}")
print(f"Distinct content_type x position_tier x freshness_tier combinations: {n_tier_combos}")


Clients in this slice: 32
Content types: 3
Distinct content_type x position_tier x freshness_tier combinations: 35


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [8]:
# 1) Does position associate with CTR? (a signal we'd expect to matter)
ctr_by_position = df.groupby("position_tier")["ctr"].mean().round(2).sort_values(ascending=False)
print("Average CTR (%) by position tier:")
print(ctr_by_position)
print()

# 2) Does word count associate with visibility?
import numpy as np
d2 = df.dropna(subset=["word_count"])
corr_wc_impr = np.corrcoef(d2["word_count"], np.log1p(d2["impressions_90d"]))[0, 1]
print(f"Correlation: word_count vs log(impressions_90d) = {corr_wc_impr:.3f}")
print()

# 3) Does content type associate with engagement?
engagement_by_type = df.groupby("content_type")["engagement_rate"].mean().round(2)
print("Average engagement_rate (%) by content_type:")
print(engagement_by_type)


Average CTR (%) by position tier:
position_tier
top_3       1.48
page_1      0.65
striking    0.32
page_3_5    0.22
deep        0.15
Name: ctr, dtype: float64

Correlation: word_count vs log(impressions_90d) = 0.337

Average engagement_rate (%) by content_type:
content_type
comparison article    0.29
feedly article        1.79
keyword article       2.65
Name: engagement_rate, dtype: float64


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

This notebook's evidence supports observed claims ("CTR is higher, on average, for pages in top_3 position in this 30,000-row slice"), directional claims ("word count shows a moderate positive association with visibility here"), and decision-support claims ("this suggests where a content team might look first"). It does not support causal claims -- nothing here was run as an experiment, so I can't say a longer article causes more impressions, only that they're associated in this data. It does not prove anything about how Google's ranking algorithm works internally -- I only see outcomes (position, clicks, impressions), never the ranking mechanism itself. It does not claim these patterns hold at full ~79M-row warehouse scale -- that has to be re-checked once I move past the 30k-row starter slice. And it makes no claims about AI rankings or citations -- ai_sessions_90d / ai_traffic_pct measure AI-referred click-throughs only, not whether or how AI tools rank or cite the content.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.